In [28]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("moltean/fruits")

print("Path to dataset files:", path)


Path to dataset files: /home/lilgreg/.cache/kagglehub/datasets/moltean/fruits/versions/87


In [ ]:
# Import libraries
import os
import cv2 as cv
import torch
import numpy as np
from ultralytics import YOLO
import matplotlib.pyplot as plt
import shutil

In [37]:


# 1. Setup your paths
src_train = os.path.join(path, 'fruits-360_100x100/fruits-360/Training') 
target_dir_train = os.path.join(path, 'cleaned_dataset/Training')
src_test = os.path.join(path, 'fruits-360_100x100/fruits-360/Test')
target_dir_test = os.path.join(path, 'cleaned_dataset/Test')

# 2. Define what to exclude (Nuts/Seeds/Misc)
exclude_list = {
    'Almonds', 'Caju seed', 'Chestnut', 'Cocos', 'Hazelnut', 
    'Nut', 'Pistachio', 'Walnut', 'Ginger Root'
}

# 3. Define the mapping for merging varieties
# Format: 'Folder Name Keyword': 'New Consolidated Name'
merge_map = {
    'Apple': 'Apple',
    'Banana': 'Banana',
    'Cherry': 'Cherry',
    'Grape': 'Grape',
    'Melon': 'Melon',
    'Pear': 'Pear',
    'Pepper': 'Pepper',
    'Potato': 'Potato',
    'Tomato': 'Tomato',
    'Peach': 'Peach',
    'Nectarine': 'Nectarine',
    'Plum': 'Plum',
    'Lemon': 'Lemon',
    'Mango': 'Mango',
    'Onion': 'Onion',
    'Cucumber': 'Cucumber',
    'Pineapple': 'Pineapple',
    'Avocado': 'Avocado'
}

# 4. Function to clean and merge dataset
def clean_and_merge_dataset(src_dir, target_dir):
    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    for folder in os.listdir(src_dir):
        folder_path = os.path.join(src_dir, folder)
        
        if not os.path.isdir(folder_path):
            continue
        
        # Check if folder should be excluded
        if any(exclude in folder for exclude in exclude_list):
            print(f"Excluding {folder}...")
            continue
        
        # Determine new folder name based on merge_map
        new_folder_name = None
        for key, value in merge_map.items():
            if key in folder:
                new_folder_name = value
                break
        
        if new_folder_name is None:
            print(f"No mapping found for {folder}, skipping...")
            continue
        
        new_folder_path = os.path.join(target_dir, new_folder_name)
        
        # Create new folder if it doesn't exist
        if not os.path.exists(new_folder_path):
            os.makedirs(new_folder_path)
        
        # Move images to the new folder
        for img_file in os.listdir(folder_path):
            src_img_path = os.path.join(folder_path, img_file)
            dst_img_path = os.path.join(new_folder_path, img_file)
            shutil.copy(src_img_path, dst_img_path)  # Use copy to keep original dataset intact

In [ ]:

clean_and_merge_dataset(src_train, target_dir_train)
clean_and_merge_dataset(src_test, target_dir_test)
train_dir = target_dir_train
test_dir = target_dir_test
model = YOLO("yolo11n-cls.pt")


No mapping found for Carrot 1, skipping...
No mapping found for Limes 1, skipping...
No mapping found for Beetroot 1, skipping...
No mapping found for Cactus fruit red 1, skipping...
Excluding Nut 3...
No mapping found for Carambola 1, skipping...
No mapping found for Cantaloupe 2, skipping...
No mapping found for Lychee 1, skipping...
Excluding Hazelnut 1...
No mapping found for Orange peeled 1, skipping...
No mapping found for Orange 2, skipping...
No mapping found for Ginger 2, skipping...
No mapping found for Carambola 2, skipping...
No mapping found for Strawberry 2, skipping...
No mapping found for Granadilla 1, skipping...
No mapping found for Blackberry 1, skipping...
No mapping found for Cauliflower 1, skipping...
No mapping found for Zucchini 1, skipping...
No mapping found for Apricot 1, skipping...
No mapping found for Raspberry 3, skipping...
No mapping found for Zucchini dark 1, skipping...
No mapping found for Cactus fruit 1, skipping...
No mapping found for Guava 1, ski

In [40]:


# train the model
results = model.train(
    data=train_dir, 
    epochs=10, 
    imgsz=100,
    hsv_s=0.1,   # Randomly vary saturation by 10%
    hsv_v=0.4,   # Randomly vary brightness by 40%
    degrees=15,  
    flipud=0.5,
    scale=0.2,   # Randomly zoom 50%-150%
    translate=0.5,  # Random translation to shift position further
    project="runs/fruit_veg_classifier",  
    name="training_v3" 
)

Ultralytics 8.4.36 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 2060, 5739MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/lilgreg/.cache/kagglehub/datasets/moltean/fruits/versions/87/cleaned_dataset/Training, degrees=15, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.1, hsv_v=0.4, imgsz=100, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=training_v3, nbs=64, nms=False, opse

In [ ]:
def predict_frame(model, frame):
    h, w, _ = frame.shape
    size = min(h, w) # Get the smallest dimension to make a square
    
    # Calculate coordinates for a center crop
    start_x = (w - size) // 2
    start_y = (h - size) // 2
    
    # Crop the image to just the center square
    cropped_frame = frame[start_y:start_y+size, start_x:start_x+size]
    
    # Resize to 100x100 to match training size
    resized_frame = cv.resize(cropped_frame, (100, 100))
    
    # Optional: Draw a box on the original frame so you know where to hold the fruit
    cv.rectangle(frame, (start_x, start_y), (start_x+size, start_y+size), (255, 0, 0), 2)

    results = model.predict(resized_frame, verbose=False)
    
    if results and len(results) > 0:
        probs = results[0].probs
        return results[0].names[probs.top1], probs.top1conf.item()
    return "None", 0.0

def boost_vibrancy(frame):
    # 1. Convert to HSV (Hue, Saturation, Value)
    hsv = cv.cvtColor(frame, cv.COLOR_BGR2HSV).astype("float32")
    
    # 2. Boost Saturation (the middle channel) by 1.5x
    hsv[:, :, 1] = hsv[:, :, 1] * 1.3
    hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255) # Keep in 0-255 range
    
    # 3. Convert back to BGR
    vibrant_frame = cv.cvtColor(hsv.astype("uint8"), cv.COLOR_HSV2BGR)
    
    # 4. Light Contrast Boost (Alpha > 1.0)
    # vibrant_frame = cv.convertScaleAbs(vibrant_frame, alpha=1.2, beta=10)
    
    return vibrant_frame

In [49]:
##
# Opens a video capture device with a resolution of 800x600
# at 30 FPS.
##
def open_camera(cam_id = 0):
    # Specify the backend (CAP_V4L2) for Linux
    cap = cv.VideoCapture(cam_id, cv.CAP_V4L2)
    
    # Check if it actually opened
    if not cap.isOpened():
        print(f"Could not open camera {cam_id}. Trying index 1...")
        cap = cv.VideoCapture(1, cv.CAP_V4L2) # Sometimes the ID shifts to 1
        
    cap.set(cv.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv.CAP_PROP_FRAME_HEIGHT, 480)
    return cap
 
##
# Gets a frame from an open video device, or returns None
# if the capture could not be made.
##
def get_frame(device):
    ret, img = device.read()
    if (ret == False): # failed to capture
        print("Failed to capture from camera. Check if it is connected and try again.")
        return None
    return img
 
##
# Closes all OpenCV windows and releases video capture device
# before exit.
##
def cleanup(device): 
    cv.destroyAllWindows()
    device.release()
 

########### Main Program ###########
if __name__ == "__main__":
    # 1. Load the model here!
    model = YOLO("runs/classify/runs/fruit_veg_classifier/training_v3/weights/best.pt")
    
    camera_id = 0
    dev = open_camera(camera_id)

    while True:
        img_orig = get_frame(dev)
        if img_orig is None:
            break
        
        # Get prediction
        img_vibrant = boost_vibrancy(img_orig)
        disp_img = img_vibrant
        pred_class, confidence = predict_frame(model, disp_img)

        # 2. Only display if confidence is high (e.g., > 50%)
        if confidence > 0.50:
            label = f"{pred_class} ({confidence*100:.1f}%)"
            # Draw the text on the image
            cv.putText(disp_img, label, (50, 50), cv.FONT_HERSHEY_SIMPLEX, 
                       1, (0, 255, 0), 2, cv.LINE_AA)
        
        cv.imshow("Fruit & Veg Detection", disp_img)

        if cv.waitKey(1) == 27: # ESC to quit
            break
 
    cleanup(dev)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is not the object's thread (0x15770930).
Cannot move to target thread (0x5152f380)

QObject::moveToThread: Current thread (0x5152f380) is n